In [ ]:
# 의료 기초 데이터 분석 미니 프로젝트
# NumPy + Pandas + Matplotlib

In [ ]:
# ==============================
# [A] 라이브러리 import
# ==============================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ==============================
# [B] pandas 출력 옵션(선택)
# ==============================
pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 200)

In [4]:
# ==============================
# [1] 데이터 준비
# ==============================

data = {
    "측정일": [
        "2026-02-01","2026-02-01","2026-02-02","2026-02-02","2026-02-03",
        "2026-02-03","2026-02-04","2026-02-04","2026-02-05","2026-02-05",
        "2026-02-06","2026-02-06","2026-02-07","2026-02-07","2026-02-08"
    ],
    "환자ID": [
        "P001","P002","P003","P004","P005",
        "P006","P007","P008","P009","P010",
        "P011","P012","P013","P014","P015"
    ],
    "성별": ["F","M","M","F","F","M","M","F","F","M","F","M","M","F","F"],
    "나이":  [25, 31, 44, np.nan, 52, 39, 28, 33, 47, 61, 29, 42, np.nan, 36, 55],
    "키":    ["162cm","175cm","168cm","160cm","158cm","180cm","172cm","165cm","170cm","177cm","159cm","174cm","169cm","163cm","157cm"],
    "몸무게": ["54kg","78kg","70kg","52kg","60kg","90kg","68kg","58kg","73kg","82kg","50kg","76kg","71kg","57kg","66kg"],
    "SBP(수축기)": [108, 135, 142, 118, 150, 130, 125, 112, 140, 160, 105, 138, 145, 120, 155],
    "DBP(이완기)": [ 70,  88,  92,  75,  95,  85,  80,  72,  90, 100,  68,  89,  96,  78,  97],
    "심박수":      [72,  84,  90,  68,  95,  78,  70,  75,  88,  92,  66,  80,  86,  73,  89],
    "공복혈당":    [92,  110, 125,  98, 130, np.nan, 105,  90, 118, 140,  88, 112, 128, 100, np.nan],
    "흡연":        ["N","Y","Y","N","N","Y","N","N","Y","Y","N","Y","Y","N","N"],
    "운동빈도(주)": [3, 1, 0, 2, np.nan, 1, 4, 2, 1, 0, 5, 1, 0, 3, np.nan],
}

df = pd.DataFrame(data)

NameError: name 'np' is not defined

In [ ]:
# ==============================
# [2] 데이터 탐색
# ==============================

print("==============================")
print("2) 데이터 탐색")
print("==============================")

print("\n[head()]")
print(df.head())

print("\n[info()] (데이터 구조 확인)")
print(df.info())

print("\n[describe()] (숫자 컬럼 요약)")
num_cols = ["나이", "SBP(수축기)", "DBP(이완기)", "심박수", "공복혈당", "운동빈도(주)"]
print(df[num_cols].describe())

In [13]:
import pandas as pd
df = pd.read_csv("proj_01.csv")
# ==============================
# [3] 데이터 정제 및 변환
# ==============================
print("==============================")
print("3) 데이터 정제 및 변환")
print("==============================")

# --------------------------------------------------------
# 3-1) 타입 변환 : 날짜 문자열 -> datetime
# --------------------------------------------------------
# pd.to_datetime(): 문자열을 날짜/시간 타입으로 변환
df["측정일"] = pd.to_datetime(df["측정일"])


# --------------------------------------------------------
# 3-2) 단위 제거 : "170cm" -> 170, "65km" -> 65
# --------------------------------------------------------
# str.replace() : 문자열 치환(여기서는 단위 제거)
# astype(int) : 숫자형으로 변환
df["키_cm"] = df["키"].str.replace("cm", "", regex=False).astype(int)
df["몸무게_kg"] = df["몸무게"].str.replace("kg", "", regex=False).astype(int)

#print(df["키_cm"])
#print(df["몸무게_kg"])

# 원본(키, 몸무게) 컬럼은 "문자열"이라 분석에 불리하드
# 보통 숫자형 컬럼(키_cm, 몸무게_kg)만 쓰는 편

# --------------------------------------------------------
# 3-3) 결측치 확인
# --------------------------------------------------------
print("\n[결측치 개수 확인]")
print(df.isnull().sum())

# --------------------------------------------------------
# 3-4) 결측치 처리(예시)
# --------------------------------------------------------
'''
결측치 처리 방식은 정답이 1개가 아니라, 상황에 따라 다름
- 나이 : 평균으로 채울
- 공복혈당 : 중앙값으로 채울(극단값 영향을 줄이고 싶을 때)
- 운동빈도 : 0으로 채움(운동 기록이 없으면 0으로 가정하는 경우)
'''

df['나이'] = df['나이'].fillna(df['나이'].mean())
df['공복혈당'] = df['공복혈당'].fillna(df['공복혈당'].median())#중앙값
df['운동빈도(주)'] = df['운동빈도(주)'].fillna(0)

# --------------------------------------------------------
# 3-5) 파생 컬럼 생성(변환)
# --------------------------------------------------------

# BMI = 몸무게(kg) / (키(m)^2)
df["키_m"] = df["키_cm"] / 100
df["BMI"] = df["몸무게_kg"] / (df["키_m"] ** 2)

# 맥박압(PP) = SBP - DBP (혈관 탄성/압력 차이를 보는 간단 지표)
df["맥박압(PP)"] = df["SBP(수축기)"] - (df["DBP(이완기)"])

# 평균동맥압("MAP") = (SBP + 2*DBP) / 3 (기초적인 참고 지표)
df["평균동맥압_MAP"] = (df["SBP(수축기)"] + 2 * df["DBP(이완기)"]) / 3

# --------------------------------------------------------
# 3-6) 범주화(카테고리 만들기) : 혈압 단계 분류
# --------------------------------------------------------
'''
기준(간단 버전)
 - 정상 : SBP < 120 그리고 DBP < 80
 - 주의(상승) : SBP 120~129 그리고 DBP < 80
- 고혈압1기 : SBP 130~139 또는 DBP 80~89
- 고혈압2기 : SBP >= 140 또는 DBP >= 80
'''

def bp_category(sbp, dbp):
    if (sbp < 120) and (dbp < 80):
        return "정상"
    elif (120 <= sbp <= 129) and (dbp < 80):
        return "주의(상승)"
    elif (130 <= sbp <= 139) or (80 <= dbp <= 89):
        return "고혈압1기"
    elif (sbp >= 140) or (dbp >= 90):
        return "고혈압2기"
    else:
        return "기타"
    
# apply() : 행 단위로 함수를 적용할 때 자주 사용
# lambda row : 익명 함수(이름 없는 함수)
# lambda 입력값 : 계산식
df["혈압분류"] = df.apply(lambda row: bp_category(row["SBP(수축기)"], row["DBP(이완기)"]), axis=1) 

# --------------------------------------------------------
# 3-7) 범주화 : BMI 단계 분류(간단 예시)
# --------------------------------------------------------
'''
BMI 분류
- 저체중 : < 18.5
- 정상 : 18.5 ~ 22.9
- 과체중 : 23.0 ~ 24.9
- 비만 : >= 25.0
'''

def bmi_category(bmi):
    if bmi < 18.5:
        return "저체중"
    elif bmi < 23.0:
        return "정상"
    elif bmi < 25.0:
        return "과체중"
    else:
        return "비만"
    
df["BMI분류"] = df["BMI"].apply(bmi_category)

#print(df["BMI분류"])
print("\n[정제/변환 결과 미리보기]")
show_cols = ["측정일","환자ID","성별","나이","키_cm","몸무게_kg","SBP(수축기)","DBP(이완기)","공복혈당","운동빈도(주)","BMI","혈압분류","BMI분류"]
print(df[show_cols].head(10))

FileNotFoundError: [Errno 2] No such file or directory: 'proj_01.csv'

In [15]:
import pandas as pd
import os
df = pd.read_csv(r"C:\Users\201\Desktop\health_data.csv")
# ==============================
# [4] 분석
# ==============================
print("==============================")
print("4) 분석")
print("==============================")

# --------------------------------------------------------
# 4-1) 최고/최저 : BMI가 가장 높은/낮은 사람
# --------------------------------------------------------
max_bmi_row = df.loc[df["BMI"].idxmax()] # 최댓값 위치 반환
min_bmi_row = df.loc[df["BMI"].idxmin()] # 최솟값 위치 반환

print("\n[BMI 최고]")
print(max_bmi_row["환자ID", "성별", "나이", "키_cm", "몸무게_kg", "BMI", "BMI분류"])
print("\n[BMI 최저]")
print(min_bmi_row["환자ID", "성별", "나이", "키_cm", "몸무게_kg", "BMI", "BMI분류"])
                  
# --------------------------------------------------------
# 4-2) 혈압 분류별 평균 지표
# --------------------------------------------------------
bp_group = df.groupby("혈압분류")[["SBP(수축기)", "DBP(이완기)", "맥박압_PP", "평균동맥압_MAP", "BMI", "공복혈당"]].mean().sort_values("SBP(수축기)", ascending=False)

print("\n[혈압분류별 평균]")
print(bp_group)

# --------------------------------------------------------
# 4-3) 성별 평균 비교
# --------------------------------------------------------
gender_mean = df.groupby("성별")[["SBP(수축기)", "DBP(이완기)", "심박수", "공복혈당", "BMI", "운동빈도(주)"]].mean()
print(gender_mean)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\201\\Desktop\\health_data.csv'

In [ ]:
# --------------------------------------------------------
# 4-4) 나이대(연령대) 만들기 + 그룹 평균
# --------------------------------------------------------
'''
나이대는 분석에서 자구 사용
- pd.cut() : 숫자를 구간으로 나눠 범주형 컬럼  
'''

bins = [0, 29, 39, 49, 59, 120]
labels = ["20대", "30대", "40대", "50대", "60대 이상"]
df["나이대"] = df.cut(df["나이"], bins=bins, labels=labels, right=True)
age_group = df.groupby("나이대")[["SBP(수축기)", "DBP(이완기)", "공복혈당", "BMI"]].mean()
print("\n[나이대별 평균]")
'''
[a, b]
왼쪽 제외 오른쪽 포함
(29, 39)
29 < 나이 <= 39
'''



NameError: name 'df' is not defined